In [1]:
import os
import math
import torch
import random
import torch.nn.functional as F

from peft import PeftModel

from models import AutoTokenizer, AutoConfig, AutoModelForCausalLM

path = "data/Dream-7B-Instruct"

config = AutoConfig.from_pretrained(path, _attn_implementation="flash_attention_2")

# config.attn_layer = []

model = AutoModelForCausalLM.from_pretrained(
    path,
    config=config,
    dtype=torch.bfloat16, 
).cuda(0)

model = PeftModel.from_pretrained(model, '/mnt/cache/code/train/ca_dlm/runs/Dream-Lora-Full_all_1e-02/checkpoint-final', device_map='auto')
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(path)

pad_id = tokenizer.pad_token_id
eos_id = tokenizer.eos_token_id
mask_id = tokenizer.mask_token_id

/mnt/cache/code/envs/dlm/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/mnt/cache/code/envs/dlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.37it/s]


In [2]:
import json
import math
import time
from tqdm import tqdm

import pynvml
pynvml.nvmlInit()


@torch.inference_mode()
def mask_diffusion(messages):

    input_ids = tokenizer.apply_chat_template(messages, return_tensors='pt')[0].cuda(0)
    prompt_ids = tokenizer.apply_chat_template(messages[:-1], add_generation_prompt=True, return_tensors='pt')[0].cuda(0)

    prompt_len = prompt_ids.size(0)

    mask = torch.rand_like(input_ids, dtype=torch.float) < 0.5
    mask[:prompt_len] = False

    labels = input_ids.clone()
    labels[~mask] = -100

    input_ids[mask] = mask_id

    start_time = time.time()

    outputs = model(
        input_ids=input_ids.unsqueeze(0), 
        is_causal=False,
        use_cache=False,
        output_attentions=True
    )

    usage_time = time.time() - start_time

    handle = pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    mem_info_used = mem_info.used / 1024 ** 2

    return 0, 0, usage_time, mem_info_used

    visible = ~mask

    pred = outputs.logits.argmax(dim=-1).squeeze(0)   
    correct = (pred == labels) & mask
    wrong = (pred != labels) & mask

    attn = torch.stack(outputs.attentions, dim=1).mean(dim=1).mean(dim=1).squeeze(0)

    score_visible = attn.masked_fill(~visible.unsqueeze(0), 0).sum(dim=-1)
    score_masked = 1 - score_visible

    score_correct = attn.masked_fill(~correct.unsqueeze(0), 0).sum(dim=-1)
    score_wrong = attn.masked_fill(~wrong.unsqueeze(0), 0).sum(dim=-1)

    margin_correct = (score_visible - score_masked).masked_select(correct).mean()
    margin_wrong = (score_correct - score_wrong).masked_select(wrong).mean()

    return margin_correct.item(), margin_wrong.item(), usage_time, mem_info_used

def load_jsonl_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data = [json.loads(line.strip()) for line in lines]
    return data

data = load_jsonl_file('data/test_10k.jsonl')

all_correct = []
all_wrong = []
all_time = []
all_mem = []

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
for d in tqdm(data[:1000]):
    correct, wrong, usage_time, mem_info_used = mask_diffusion(d['messages'])
    if not math.isnan(correct) and not math.isnan(wrong):
        all_correct.append(correct)
        all_wrong.append(wrong)
        all_time.append(usage_time)
        all_mem.append(mem_info_used)   

print("Average margin for correct predictions:", sum(all_correct) / len(all_correct))
print("Average margin for wrong predictions:", sum(all_wrong) / len(all_wrong))
print("Average usage time:", sum(all_time) / len(all_time))
print("Average GPU memory usage:", sum(all_mem) / len(all_mem))

100%|██████████| 1000/1000 [00:49<00:00, 20.40it/s]

Average margin for correct predictions: 0.0
Average margin for wrong predictions: 0.0
Average usage time: 0.04617342185974121
Average GPU memory usage: 38105.854
